# EDA — Give Me Some Credit

Objetivo: entender la distribución del target, calidad de los datos (missing, outliers) y detectar posible target leakage antes de modelar.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

df = pd.read_csv('../data/raw/cs-training.csv', index_col=0)
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

## 1. Distribución del target

`SeriousDlqin2yrs` = 1 si el cliente tuvo un impago grave (90+ días de retraso) en los siguientes 2 años.

In [ ]:
target = 'SeriousDlqin2yrs'

counts = df[target].value_counts()
pcts = df[target].value_counts(normalize=True) * 100

print(counts)
print()
print(pcts.round(2))

fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(x=target, data=df, ax=ax)
ax.set_title('Distribución del target')
for i, v in enumerate(counts):
    ax.text(i, v + 1000, f'{v:,}\n({pcts[i]:.1f}%)', ha='center')
plt.show()

**Nota:** este dataset suele tener ~6-7% de impagos. Es un desbalanceo moderado, no extremo — lo tendremos en cuenta en el modelado (class weights), pero no hace falta SMOTE ni oversampling agresivo.

## 2. Missing values

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'missing': missing, 'pct': missing_pct})
missing_df[missing_df['missing'] > 0].sort_values('pct', ascending=False)

Normalmente solo hay missing en `MonthlyIncome` (~20%) y `NumberOfDependents` (~2.5%). Lo trataremos en feature engineering (no lo imputamos aquí todavía, eso se hace dentro del pipeline para evitar leakage entre train/test).

## 3. Estadísticos descriptivos y outliers

In [ ]:
df.describe().T

Presta especial atención a:
- `age`: ¿hay algún 0? (dato imposible, hay que limpiarlo)
- `RevolvingUtilizationOfUnsecuredLines`: debería estar entre 0 y ~1-2, pero a veces hay valores absurdos (miles)
- Las columnas `NumberOfTime30-59...`, `NumberOfTime60-89...`, `NumberOfTimes90DaysLate`: suelen tener valores centinela como 96 o 98, que no son retrasos reales sino códigos de error del sistema origen

In [ ]:
# Revisar valores centinela / outliers en las columnas de retrasos
delinquency_cols = [c for c in df.columns if 'Time' in c]

for col in delinquency_cols:
    print(col)
    print(df[col].value_counts().sort_index(ascending=False).head(5))
    print()

In [ ]:
# Edad: comprobar valores imposibles
print('Edad mínima:', df['age'].min())
print('Filas con age == 0:', (df['age'] == 0).sum())

In [ ]:
# Distribuciones de las variables numéricas clave
cols_to_plot = ['RevolvingUtilizationOfUnsecuredLines', 'age', 'DebtRatio', 'MonthlyIncome']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flatten(), cols_to_plot):
    # recortamos al percentil 99 solo para visualizar mejor
    data = df[col].dropna()
    upper = data.quantile(0.99)
    sns.histplot(data[data <= upper], bins=50, ax=ax)
    ax.set_title(f'{col} (recortado al p99)')
plt.tight_layout()
plt.show()

## 4. Variables numéricas vs target (análisis bivariante)

Comparamos la distribución de cada variable entre clientes que impagaron (1) y los que no (0).

In [ ]:
features = [c for c in df.columns if c != target]

fig, axes = plt.subplots(4, 3, figsize=(16, 16))
for ax, col in zip(axes.flatten(), features):
    data = df[[col, target]].dropna()
    upper = data[col].quantile(0.99)
    data = data[data[col] <= upper]
    sns.boxplot(x=target, y=col, data=data, ax=ax)
    ax.set_title(col, fontsize=9)
plt.tight_layout()
plt.show()

## 5. Matriz de correlación

Sirve para detectar multicolinealidad entre features (relevante sobre todo para el modelo logístico/WOE) y para una primera idea de qué variables se relacionan más con el target.

In [ ]:
corr = df.corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
plt.title('Matriz de correlación')
plt.show()

In [ ]:
# Correlación de cada feature con el target, ordenada
corr[target].drop(target).sort_values(key=abs, ascending=False)

## 6. Chequeo de target leakage

Target leakage = una variable que en la práctica solo se conocería *después* de saber si el cliente impagó (o que está calculada usando información del futuro). En este dataset en particular no hay columnas tan obvias como en otros (ej. 'fecha de impago'), pero conviene verificar:

- Que ninguna columna tenga correlación sospechosamente alta (>0.9) con el target — sería señal de que codifica el target de otra forma.
- Que todas las variables sean conocidas en el momento de la solicitud del crédito (no información posterior).

In [ ]:
high_corr = corr[target].drop(target)[corr[target].drop(target).abs() > 0.5]
print('Variables con correlación sospechosamente alta (>0.5):')
print(high_corr if len(high_corr) > 0 else 'Ninguna — buena señal, no hay leakage evidente por correlación.')

## 7. Resumen de hallazgos y plan de limpieza

Rellena esto a medida que ejecutes el notebook con tus propios datos:

- [ ] % de impagos en el target: ___
- [ ] Missing en `MonthlyIncome`: ___ % → estrategia: imputar con mediana (o un modelo simple) dentro del pipeline, y añadir un flag `MonthlyIncome_missing`
- [ ] Missing en `NumberOfDependents`: ___ % → estrategia: imputar con 0 o moda
- [ ] `age == 0`: ___ filas → eliminar (dato imposible) o imputar con mediana
- [ ] Valores centinela (96/98) en columnas de retrasos: ___ → tratar como categoría aparte o capear
- [ ] `RevolvingUtilizationOfUnsecuredLines` con outliers extremos: ___ → capear al percentil 99
- [ ] Variables más correlacionadas con target: ___
- [ ] Leakage detectado: Sí/No